# 🌱 Module 5: Vegetation Index Analysis

Welcome to **Module 5** of the **Deforestation Detection from Satellite Images** project!

### 🎯 Learning Objectives
1. **NDVI (Normalized Difference Vegetation Index)**: Calculate and interpret NDVI using Red and NIR bands.
2. **EVI (Enhanced Vegetation Index)**: Correct for atmospheric aerosol scattering and canopy saturation.
3. **SAVI (Soil-Adjusted Vegetation Index)**: Suppress bright soil background reflectance in partial clearings.
4. **NDWI (Normalized Difference Water/Moisture Index)**: Monitor canopy liquid water content.
5. **NBR (Normalized Burn Ratio)**: Detect fire scars and slash burns.
6. **Build Practical Workflow**: $\text{Satellite Image} \rightarrow \text{NDVI Calculation} \rightarrow \text{Vegetation Map}$ to distinguish vegetation from non-vegetation.

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import rasterio

# Add project root to sys.path
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from modules.module_05_vegetation_indices.vegetation_indices import (
    VegetationIndexCalculator,
    VegetationClassifier
)

%matplotlib inline
print("✅ Module 5 components loaded successfully!")

## 1. Load Multispectral Satellite Scene

Let's load the 5-band satellite scene: Blue, Green, Red, NIR, SWIR.

In [ ]:
sample_b_path = "../dataset/train/before/scene_train_001.tif"
with rasterio.open(sample_b_path) as src:
    bands = src.read()

print(f"Loaded satellite scene shape: {bands.shape} (Bands, Height, Width)")

## 2. Compute All Remote Sensing Indices

We calculate:
- **NDVI**: $\frac{\text{NIR} - \text{Red}}{\text{NIR} + \text{Red}}$
- **EVI**: $2.5 \times \frac{\text{NIR} - \text{Red}}{\text{NIR} + 6 \cdot \text{Red} - 7.5 \cdot \text{Blue} + 1}$
- **SAVI**: $\frac{\text{NIR} - \text{Red}}{\text{NIR} + \text{Red} + 0.5} \times 1.5$
- **NDWI**: $\frac{\text{NIR} - \text{SWIR}}{\text{NIR} + \text{SWIR}}$
- **NBR**: $\frac{\text{NIR} - \text{SWIR}}{\text{NIR} + \text{SWIR}}$

In [ ]:
calc = VegetationIndexCalculator(bands)
indices = calc.compute_all_indices()

for name, idx in indices.items():
    print(f"{name:<15}: Min = {idx.min():.3f}, Max = {idx.max():.3f}, Mean = {idx.mean():.3f}")

## 3. Compare Spectral Indices Side-by-Side

Visualizing the response across each index reveals canopy structure, moisture, and background soil suppression.

In [ ]:
def make_rgb(b):
    def s(arr):
        p2, p98 = np.percentile(arr, (2, 98))
        return np.clip((arr - p2) / (p98 - p2 + 1e-8), 0.0, 1.0)
    return np.stack([s(b[2]), s(b[1]), s(b[0])], axis=-1)

fig, axes = plt.subplots(2, 3, figsize=(16, 10), constrained_layout=True)
axes[0, 0].imshow(make_rgb(bands))
axes[0, 0].set_title("True Color (RGB)", fontweight="bold")
axes[0, 0].axis("off")

im1 = axes[0, 1].imshow(indices["NDVI"], cmap="RdYlGn", vmin=-0.2, vmax=0.9)
axes[0, 1].set_title("NDVI", fontweight="bold")
axes[0, 1].axis("off")
fig.colorbar(im1, ax=axes[0, 1], fraction=0.046, pad=0.04)

im2 = axes[0, 2].imshow(indices["EVI"], cmap="YlGn", vmin=0.0, vmax=1.0)
axes[0, 2].set_title("EVI (Atmosphere & Soil Corrected)", fontweight="bold")
axes[0, 2].axis("off")
fig.colorbar(im2, ax=axes[0, 2], fraction=0.046, pad=0.04)

im3 = axes[1, 0].imshow(indices["SAVI"], cmap="YlGn", vmin=0.0, vmax=0.9)
axes[1, 0].set_title("SAVI (Soil-Adjusted)", fontweight="bold")
axes[1, 0].axis("off")
fig.colorbar(im3, ax=axes[1, 0], fraction=0.046, pad=0.04)

im4 = axes[1, 1].imshow(indices["NDWI_Moisture"], cmap="BrBG", vmin=-0.6, vmax=0.8)
axes[1, 1].set_title("NDWI Moisture (Leaf Water)", fontweight="bold")
axes[1, 1].axis("off")
fig.colorbar(im4, ax=axes[1, 1], fraction=0.046, pad=0.04)

im5 = axes[1, 2].imshow(indices["NBR"], cmap="copper", vmin=-0.4, vmax=0.8)
axes[1, 2].set_title("NBR (Burn Ratio)", fontweight="bold")
axes[1, 2].axis("off")
fig.colorbar(im5, ax=axes[1, 2], fraction=0.046, pad=0.04)

plt.show()

## 4. Practical Roadmap Workflow: Satellite Image $\rightarrow$ NDVI $\rightarrow$ Vegetation Map

We categorize NDVI into discrete land-cover bins:
- **Water**: $\text{NDVI} < 0.0$
- **Bare Soil / Cleared**: $0.0 \le \text{NDVI} < 0.25$
- **Degraded / Edge**: $0.25 \le \text{NDVI} < 0.55$
- **Dense Forest**: $\text{NDVI} \ge 0.55$

In [ ]:
classifier = VegetationClassifier(forest_threshold=0.50)
ndvi = indices["NDVI"]
veg_map = classifier.create_vegetation_map(ndvi)

from matplotlib.colors import ListedColormap
cmap_custom = ListedColormap(["#2980b9", "#d35400", "#f1c40f", "#27ae60"])

fig, axes = plt.subplots(1, 3, figsize=(16, 5.5), constrained_layout=True)
axes[0].imshow(make_rgb(bands))
axes[0].set_title("1. Satellite Image (RGB)", fontweight="bold")
axes[0].axis("off")

im_n = axes[1].imshow(ndvi, cmap="RdYlGn", vmin=-0.2, vmax=0.9)
axes[1].set_title("2. NDVI Calculation", fontweight="bold")
axes[1].axis("off")
fig.colorbar(im_n, ax=axes[1], fraction=0.046, pad=0.04)

im_m = axes[2].imshow(veg_map, cmap=cmap_custom, vmin=0, vmax=3)
axes[2].set_title("3. Classified Vegetation Map", fontweight="bold", color="darkgreen")
axes[2].axis("off")
cbar2 = fig.colorbar(im_m, ax=axes[2], fraction=0.046, pad=0.04, ticks=[0.375, 1.125, 1.875, 2.625])
cbar2.ax.set_yticklabels(["Water", "Soil/Cleared", "Degraded", "Dense Forest"], fontweight="bold")

plt.show()

## 5. Quantitative Land Cover & Forest Area Breakdown

Quantify the exact forest canopy area in hectares ($1\text{ pixel at 10m} = 0.01\text{ ha}$).

In [ ]:
stats = classifier.compute_class_statistics(veg_map)
for c_name, s in stats.items():
    print(f"{c_name:<25}: {s['pixel_count']:>6} pixels | {s['percentage']:>6.2f}% | {s['area_ha']:>8.2f} ha")